# Value Functions

## Explore MDP

In [ ]:
from dynamic_programming import RabbitMDP

In [ ]:
mdp = RabbitMDP()

In [ ]:
mdp.STATES

In [ ]:
mdp.ACTIONS

In [ ]:
mdp.REWARDS

Not all actions allowed in all states:

In [ ]:
for s in mdp.STATES:
    print(s)
    for a in mdp.A(s):
        print('  ', a)

## p function

How does p function work:

In [ ]:
mdp.p('hungry', 'eat', 'eating', 1)

In [ ]:
mdp.p('hungry', 'eat', 'dead', -1)

Not allowed transitions have p =0

In [ ]:
mdp.p('hungry', 'eat', 'eating', 0)

In [ ]:
mdp.p('hungry', 'eat', 'idle', 1)

For state, action pair, show p for all possible transitions.

Sum up to 1!

In [ ]:
for s in mdp.STATES:
    for r in mdp.REWARDS:
        print(mdp.p('hungry', 'eat', s, r))

## V table

Let's create an initial table with all 0 for all states

Dictionary

In [ ]:
V = {state: 0 for state in mdp.STATES}
V

## Action value function $q$

$$ q(s,a) = \sum_{s'} \sum_r p(s',r|s,a)[r + \gamma \mathop{v_\pi}(s')] $$

Need to define gamma first

In [ ]:
gamma = 0.9

Take an example transition

In [ ]:
s = 'hungry'
a = 'eat'
s_next = 'eating'
r = 1
mdp.p(s, a, s_next, r)

Value estimate of next state = 0 because table is all 0

In [ ]:
V[s_next]

So last part of definition is equal to:

In [ ]:
r + gamma * V[s_next]

In [ ]:
mdp.p(s, a, s_next, r) * (r + gamma * V[s_next])

Now take the other transition for this state-action pair

In [ ]:
s_next = 'dead'
r = -1
mdp.p(s, a, s_next, r)

In [ ]:
V[s_next]

In [ ]:
r + gamma * V[s_next]

In [ ]:
mdp.p(s, a, s_next, r) * (r + gamma * V[s_next])

In [ ]:
0.8 + -0.2

Let's compute this for all transition possibilities

In [ ]:
s = 'hungry'
a = 'eat'
q = 0
for s_next in mdp.STATES:
    for r in mdp.REWARDS:
        q += mdp.p(s, a, s_next, r) * (r + gamma * V[s_next])
print(q)

Make that a function

In [ ]:
def q(s, a):
    value = 0
    for s_next in mdp.STATES:
        for r in mdp.REWARDS:
            value += mdp.p(s, a, s_next, r) * (r + gamma * V[s_next])
    return value

In [ ]:
q('hungry', 'eat')

Other possible transition from same state

In [ ]:
q('hungry', 'stay') # 0.1 chance of dead with -1 reward

In [ ]:
q('hungry', 'wakeup')

## Policy

Create a policy $\pi$ randomly distributes

In [ ]:
pi = {s: {a: 1/len(mdp.A(s)) for a in mdp.A(s)} for s in mdp.STATES}
pi

In [ ]:
s = 'hungry'
a = 'eat'
pi[s][a]

In [ ]:
for a in mdp.A(s):
    print(a, pi[s][a])

## State value function $v$

$$ v(s) = \sum_a \mathop{\pi}(a|s) q(s,a) $$

In [ ]:
s = 'hungry'
a = 'eat'
q(s,a)

In [ ]:
pi[s][a] * q(s,a)

In [ ]:
s = 'hungry'
a = 'stay'
pi[s][a] * q(s,a)

In [ ]:
0.3 + -0.05

Let's implement a function that computes this

In [ ]:
def v_pi(s):
    value = 0
    for a in mdp.A(s):
        value += pi[s][a] * q(s,a)
    return value

In [ ]:
v_pi('hungry')

In [ ]:
v_pi('eating')

In [ ]:
v_pi('idle')

Why is this all 0?
reward of only possible transition = 0
V for next state (hungry) is still 0

In [ ]:
q('idle', 'wakeup') # r = 0 and v[hungry] = 0

In [ ]:
v_pi('dead')

## Policy evaluation

Let's update the entire table for all states

In [ ]:
for s in mdp.STATES:
    V[s] = v_pi(s)
V

In [ ]:
q('idle', 'wakeup')

In [ ]:
s = 'idle'
a = 'wakeup'
pi[s][a]

In [ ]:
mdp.p(s,a, 'hungry', 0)

In [ ]:
V['hungry']

In [ ]:
gamma * V['hungry']

In [ ]:
V

In [ ]:
q('hungry', 'eat')

Now do another run and see how it changes again.

In [ ]:
delta = 0
for s in mdp.STATES:
    old = V[s]
    V[s] = v_pi(s)
    delta = max(abs(V[s] - old), delta)
print(delta)
V

Implement this in a function

In [ ]:
def update_V():
    delta = 0
    for s in mdp.STATES:
        old = V[s]
        V[s] = v_pi(s)
        delta = max(abs(V[s] - old), delta)
    return delta

Reset value table and repeat

In [ ]:
V = {state: 0 for state in mdp.STATES}
V

In [ ]:
delta = update_V()
print(delta)
V

Put this in a loop that stops until delta ~= 0

In [ ]:
def converge_V():
    delta = update_V()
    while delta > 1e-16:
        delta = update_V()

In [ ]:
converge_V()
V

This is the exact value function for this policy (i.e. completely random)

In [ ]:
pi

Look at action values

In [ ]:
for s in mdp.STATES:
    print(s)
    for a in mdp.A(s):
        print('  ', a, q(s,a))

## Policy improvement

Let's act greedy and only take actions that have maximum value

In [ ]:
pi['hungry']['eat'] = 1
pi['hungry']['stay'] = 0
pi['eating']['eat'] = 1
pi['eating']['return'] = 0
pi

How does that influence our new V

In [ ]:
v_pi('eating')

In [ ]:
converge_V()
V

In [ ]:
for s in mdp.STATES:
    print(s)
    for a in mdp.A(s):
        print('  ', a, q(s,a))

You can see tha the "return" action from eating now has slightly higher value than "eat". Change policy again

In [ ]:
pi['eating']['eat'] = 0
pi['eating']['return'] = 1
pi

In [ ]:
v_pi('eating')

In [ ]:
converge_V()
V

In [ ]:
for s in mdp.STATES:
    print(s)
    for a in mdp.A(s):
        print('  ', a, q(s,a))

No need to change policy. If acting greedy, it is still the same

## Policy evaluation convergence
Test to see if V really converges to the value of pi

Keep policy the same, but reset V

In [ ]:
V = {state: 0 for state in mdp.STATES}
V

In [ ]:
converge_V()
V

Yup, really converges. So this is actual value for this policy, and this policy can't be improved anymore, it already acts greedy. So is this now the optimal policy??

## Greedy with NumPy

In [ ]:
import numpy as np

In [ ]:
s = 'hungry'
{a: q(s, a) for a in mdp.A(s)}

In [ ]:
[q(s,a) for a in mdp.A(s)]

In [ ]:
np.max([q(s,a) for a in mdp.A(s)])

In [ ]:
max_a = np.argmax([q(s,a) for a in mdp.A(s)])
max_a

In [ ]:
s = 'eating'
max_a = np.argmax([q(s,a) for a in mdp.A(s)])
max_a

In [ ]:
mdp.A(s)[max_a]

In [ ]:
np.max([q(s,a) for a in mdp.A(s)])

In [ ]:
V[s]

So, in state 'eating' taking the action with the highest expected return is equal to the value of the state. Then this policy is optimal and V represents v*